Copyright 2021 The TensorFlow Authors.

#Assess privacy risks with TensorFlow Privacy Secret Sharer Attack

In this colab, we demonstrate how [secret sharer](https://arxiv.org/abs/1802.08232) works on a language model. We will train a model with "secrets", i.e. random sequences, inserted in the training data, and then evaluate if the model has "memorized" those secrets.

(The code is largely based on a Tensorflow tutorial for training a character-level LSTM on Shakespeare dataset.)

In [ ]:

%pip install tf-models-official
#!pip install git+https://github.com/tensorflow/privacy.git
#clear_output()

In [ ]:
import functools
import os
import numpy as np
import tensorflow as tf


from tensorflow_privacy.privacy.privacy_tests.secret_sharer.generate_secrets import SecretConfig, TextSecretProperties, construct_secret
from tensorflow_privacy.privacy.privacy_tests.secret_sharer.exposures import compute_exposure_interpolation, compute_exposure_extrapolation

# Define functions

In [ ]:


SEQ_LENGTH = 1024

TRAIN_EPOCHS = 4
BATCH_SIZE = 32
DATASET_SIZE = 20000




In [ ]:
pip install pandas

In [ ]:
import pandas as pd
import csv
def load_text_from_csv(path_to_file):
    df = pd.read_csv(path_to_file, usecols=["TEXT"], dtype=str, encoding="ISO-8859-1", quoting=csv.QUOTE_NONE) 
    text = " ".join(df["TEXT"].dropna().tolist()) 
    return text

In [ ]:
#@title Function for getting dataset
def get_dataset(filename="NOTEEVENTS_deanonymized.csv",
                batch_size=None, seq_length=SEQ_LENGTH):
  """Creates a dataset from a given text file.

  Args:

    batch_size: Batch size to use.
    seq_length: The length of the LSTM sequence.

  Returns:
    A tuple, consisting of the Dataset and the class to character mapping
    and character to class mapping.
  """
  tokenizer = AutoTokenizer.from_pretrained("gpt2")
  tokenizer.pad_token = tokenizer.eos_token
  text = load_text_from_csv(filename)


  def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text


  tokenizer = AutoTokenizer.from_pretrained("gpt2")
  tokenizer.pad_token = tokenizer.eos_token
  input_ids = tokenizer.encode(text, return_tensors="tf")[0]
  seq_length = 1024  # GPT-2 uses max sequence of 1024 tokens
  dataset = tf.data.Dataset.from_tensor_slices(input_ids)
  dataset = dataset.batch(seq_length + 1, drop_remainder=True)
  def split_input_target(chunk):
    return chunk[:-1], chunk[1:]

  dataset = dataset.map(split_input_target)
  dataset = dataset.shuffle(1000)
    
  train_dataset = dataset.take(DATASET_SIZE).batch(BATCH_SIZE, drop_remainder=True)
  val_dataset = dataset.skip(DATASET_SIZE).batch(BATCH_SIZE, drop_remainder=True)
  return train_dataset, val_dataset

In [ ]:
import os
import tensorflow as tf
from transformers import TFAutoModelForCausalLM, AutoTokenizer, AdamWeightDecay
from tensorflow.keras.optimizers import SGD
from tensorflow_privacy.privacy.optimizers.dp_optimizer_keras import DPKerasSGDOptimizer
from tensorflow.keras.callbacks import History

def train_model(dataset, validation_dataset, batch_size, vocab_size, train_epochs, checkpoint_dir, dp):
    """Trains Transformer model with checkpoints."""

    train_steps = len(dataset) // batch_size 
    print("train steps", train_steps)
    model = TFAutoModelForCausalLM.from_pretrained("gpt2")
    noise_multiplier = 5
    l2_norm_clip = 1e-6
    num_microbatches = BATCH_SIZE
    if dp:
        optimizer = DPKerasSGDOptimizer(
            learning_rate=0.03,
            noise_multiplier=noise_multiplier,
            l2_norm_clip=l2_norm_clip,
            num_microbatches=1
        )
        print("Using DP-SGD with noise_multiplier =", noise_multiplier)
    else:
        optimizer = tf.keras.optimizers.SGD(learning_rate=0.03)
    
    loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)


    model.compile(optimizer=optimizer, loss=loss)



    callbacks = []

    if checkpoint_dir:
        checkpoint_filepath = os.path.join(checkpoint_dir, 'ckpt_{epoch}')
        checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_filepath,
            save_weights_only=True,
            save_best_only=True,  # Saves best model based on loss
            monitor="loss",  
            mode="min",
            verbose=1
        )
        callbacks.append(checkpoint_callback)
        callbacks.append(History())

        checkpoint_latest = tf.train.latest_checkpoint(checkpoint_dir)
        if checkpoint_latest:
            model.load_weights(checkpoint_latest)
            initial_epoch = int(os.path.basename(checkpoint_latest).split('_')[-1])
            print(f'Checkpoint found: {checkpoint_latest}. Resuming from epoch {initial_epoch}.')
        else:
            initial_epoch = 0
    else:
        initial_epoch = 0


    history = model.fit(
        dataset,
        epochs=train_epochs,
        validation_data=validation_dataset,
        initial_epoch=initial_epoch,
        steps_per_epoch=train_steps,
         callbacks=callbacks,
        verbose=2
    )

    return history, callbacks

In [ ]:
#history.history['loss']

In [ ]:
from transformers import TFAutoModelForCausalLM, AutoModelForCausalLM
import tensorflow as tf


def load_gpt2_model(model_checkpoint, checkpoint_dir, dp, pytorch=False):
    """Loads GPT-2 with weights from previous checkpoint."""



    prediction_model = TFAutoModelForCausalLM.from_pretrained(model_checkpoint)
    prediction_model.config.save_pretrained(checkpoint_dir)
    if dp:
        noise_multiplier = 5   #DP for Epsilon 1.5, delta= dataset_size^-1
        l2_norm_clip = 1e-6
        num_microbatches = BATCH_SIZE
        optimizer = DPKerasSGDOptimizer(
            learning_rate=0.03,
            noise_multiplier=noise_multiplier,
            l2_norm_clip=l2_norm_clip,
            num_microbatches=1
        )
        print("Usando DP-SGD con noise_multiplier =", noise_multiplier)
    else:
        optimizer = tf.keras.optimizers.SGD(learning_rate=0.03)
    

    latest_checkpoint = tf.train.latest_checkpoint(checkpoint_dir)
    if latest_checkpoint:
        prediction_model.load_weights(latest_checkpoint)
        print(f'Pesos cargados desde {latest_checkpoint}')
    else:
        print('No se encontró ningún checkpoint, cargando pesos preentrenados.')


    prediction_model.compile(optimizer=optimizer, loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))

    return prediction_model


In [ ]:
import csv
from tqdm import tqdm
from itertools import islice
import json
import re
import numpy as np


def allSecretTypesDone(secrets, amount_secrets) -> bool:
    for key in secrets.keys():
        if (amount_secrets == -1 or len(secrets[key]) <= amount_secrets):
            return False
    return True

def read_secrets(input_csv, amount_secrets, maxRepetitions=20, minRepetitions=10):
    secrets = {"yearData":[], "stateData":[], "ageData":[], "doctorData":[],
               "hospitalData":[], "hospitalUnitData":[], "dateData":[], 
               "locationData":[], "numberData":[], "firstNameData":[],"phoneData":[],
              "monthData":[], "countryData":[], "genericNumberData":[] }
    

    
    with open(input_csv, mode='r', encoding='utf-8') as infile:
        reader = csv.DictReader(infile)  
        for i, row in enumerate(tqdm(reader)):
            repetitions = row['repetitions']
            if (repetitions != "repetitions" and int(repetitions)<maxRepetitions and int(repetitions)>minRepetitions):
                phrase = row['phrase']
                secretType = row['type']
                secrets[secretType].append({
                    'repetitions': int(repetitions), 
                    'phrase': phrase,
                    'type':secretType
                })
                if allSecretTypesDone(secrets, amount_secrets):
                    print("allSecretTypesDone is done")
                    break

    for key in secrets.keys():
        secrets[key] = secrets[key][0:amount_secrets]
        
    return secrets


    
def generate_text_secrets(input_csv, amount_secrets, maxRepetitions=50, minRepetitions=1):
    
    secrets = read_secrets(input_csv=input_csv, amount_secrets=amount_secrets, maxRepetitions=maxRepetitions, minRepetitions= minRepetitions)
    print(len(secrets),"secrets loaded")
    for key in secrets.keys():
        print(key, len(secrets[key]))
    all_reference_secrets = []
    for secretKey in tqdm(secrets.keys()):
        for secret in secrets[secretKey]:
          secret["datasets"] = {}
          r = secret["repetitions"]
          seqs = secret["phrase"]
          sType = secret["type"]
    
          indexed_seqs = np.array([tokenizer.encode(seqs, padding='max_length', max_length=SEQ_LENGTH)])
          secret["datasets"][r] = {
            'data': tf.data.Dataset.from_tensor_slices(indexed_seqs).map(lambda chunk: chunk).batch(1),
            'type':sType
          }
          
   
    return secrets





## Calculating perplexity on GPT2 finetuned with MIMIC-III

With context consisting of random phrases

In [ ]:
from tensorflow.nn import softmax
import numpy as np
import tensorflow as tf
from tqdm import tqdm
import random

def read_phrasePool(input_csv, context_size, context_length_bin=0):
    secrets = {"yearData":[], "stateData":[], "ageData":[], "doctorData":[],
               "hospitalData":[], "hospitalUnitData":[], "dateData":[], 
               "locationData":[], "numberData":[], "firstNameData":[],"phoneData":[],
              "monthData":[], "countryData":[], "genericNumberData":[] }

    context_length_bins = [{"min":0, "max":1024}, 
                          {"min":1, "max":5},
                          {"min":6, "max":10},
                          {"min":11, "max":20},
                          {"min":21, "max":40},
                          {"min":41, "max":1024}]
                          
    with open(input_csv, mode='r', encoding='utf-8') as infile:
        reader = csv.DictReader(infile)  
        for i, row in enumerate(tqdm(reader)):
            phrase = row['phrase']
            if ((len(phrase) >= context_length_bins[context_length_bin]["min"])
            and (len(phrase) <= context_length_bins[context_length_bin]["max"])):
                secretType = row['type']
                secrets[secretType].append({ 
                    'phrase': phrase
                })
                if allSecretTypesDone(secrets, context_size):
                    print("allPhraseTypesDone is done")
                    break

    for key in secrets.keys():
        secrets[key] = secrets[key][0:context_size]
        
    return secrets

    
def sample_context(contexts, secretKey):
    potential_context = random.choice(contexts[secretKey])  #Assuming context is a global variable resulted of read_phrasePool
    if len(potential_context) >= 1000:  #Context window of GPT-2 is 1024 and it must take context+hint+secret
        return sample_context(contexts,secretKey)
    return potential_context
    
def compute_perplexity_with_context_for_secret(secretKey, secrets, checkpoint_dir, dp=False, context_size=500, context_length_bin=0,
                                               prob_sampling = 1, amount_context=100):
    """Get perplexity for given examples, using random contexts.
    There are 5 bins depending on the length of the context (in tokens), 
    context_length_bin:  0 = all contexts are used
                         1 =  1-5 tokens
                         2 = 6-10 tokens
                         3 = 11-20 tokens
                         4 = 21-40 tokens
                         5 = 40+ tokens

    Args:
        secrets: a list, where each element is a secret dictionary
        checkpoint_dir: the directory from which to load checkpoints
        vocab_size: vocabulary size
        allowedVocab: Array of possible word that the model can output (masks the predicted distribution)
        dp: If set to True, load_gpt2_model loads DP-SGD optimizer instead of SGD.
        context_size: Size of context pool from where we sample one for each secret.
        context_length_bin: {0,1,2,3,4,5}, explained on previous section.
        prob_sampling: [0,1], probability of using context.
        context_amount: Amount of different contexts used for each secret.
    Returns:
        Perplexity for the input examples.
    """
    contexts = read_phrasePool("phrase_context_pool.csv", context_size, context_length_bin)
    prediction_model = load_gpt2_model("gpt2", checkpoint_dir, dp)

    cce = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE) #NOT USED
    sum_prob_allsecrets = 0
    secret_amount = 0
    for i, secret in enumerate(secrets):
        for r in secret["datasets"]:
            d = secret["datasets"][r]
            if (d['type'] != secretKey):
                print("Secret type", secret["datasets"][r]["type"], "found, skipping")
                break
            context = ""
            rand_number = random.randint(0,9)
            if (rand_number < (prob_sampling*10)):
                context = sample_context(contexts, secret["datasets"][r]["type"])["phrase"]
            tokenized_context = tokenizer.encode(context, add_special_tokens=False)
            context_len = len(tokenized_context)
            indexed_seqs = d['data'] 
            perplexity_for_secret = []
            secrets_sum_prob = 0
            for seq_batch in indexed_seqs:
                for seq in seq_batch:
                    secret_amount += 1
                    maxLength = 1024-context_len
                    full_input = tokenized_context + (seq.numpy().tolist()[:maxLength])
                    tokens = tf.convert_to_tensor([full_input]) 
                    logits = prediction_model.predict(tokens).logits
                    predictions_prob = softmax(logits, axis=-1)
                    token_probs = 0
                    realLength = 0 #Seq has padding, repeats 2056 after the phrase ends, with RealLength we count tokens to get real phrase length
                    for k in range(context_len + 1, len(full_input)): 
                        token = full_input[k]  # Current token
                        prob = predictions_prob[0, k-1, token].numpy()  # Probability for token i given previous tokens
                        token_probs -= np.log2(prob)
                        print("token_probs",token_probs)
                        realLength += 1
                        if (token == 50256):
                            break
                    secret_prob = np.sqrt(2**(-token_probs)) #2 es la base de SparseCategoricalCrossentropy
                    print("secret_prob",secret_prob)
                    secrets_sum_prob += secret_prob

    print("secrets_sum_prob",secrets_sum_prob)
    print("secret_amount", secret_amount)
    inverse_secret_size = (1 / np.sqrt(secret_amount))    
    print("inverse_secret_size",inverse_secret_size)
    divergence = -2*np.log2(inverse_secret_size*secrets_sum_prob) 
    return divergence





In [ ]:
from tensorflow.nn import softmax
import numpy as np
import tensorflow as tf
from tqdm import tqdm
import random

def read_phrasePool(input_csv, context_size, context_length_bin=0):
    secrets = {"yearData":[], "stateData":[], "ageData":[], "doctorData":[],
               "hospitalData":[], "hospitalUnitData":[], "dateData":[], 
               "locationData":[], "numberData":[], "firstNameData":[],"phoneData":[],
              "monthData":[], "countryData":[], "genericNumberData":[] }

    context_length_bins = [{"min":0, "max":1024}, 
                          {"min":1, "max":5},
                          {"min":6, "max":10},
                          {"min":11, "max":20},
                          {"min":21, "max":40},
                          {"min":41, "max":1024}]
                          
    with open(input_csv, mode='r', encoding='utf-8') as infile:
        reader = csv.DictReader(infile)  
        for i, row in enumerate(tqdm(reader)):
            phrase = row['phrase']
            if ((len(phrase) >= context_length_bins[context_length_bin]["min"])
            and (len(phrase) <= context_length_bins[context_length_bin]["max"])):
                secretType = row['type']
                secrets[secretType].append({ 
                    'phrase': phrase
                })
                if allSecretTypesDone(secrets, context_size):
                    print("allPhraseTypesDone is done")
                    break

    for key in secrets.keys():
        secrets[key] = secrets[key][0:context_size]
        
    return secrets

    
def sample_context(contexts, secretKey):
    potential_context = random.choice(contexts[secretKey])  #Assuming context is a global variable resulted of read_phrasePool
    if len(potential_context) >= 1000:  #Context window of GPT-2 is 1024 and it must take context+hint+secret
        return sample_context(contexts,secretKey)
    return potential_context
    
def compute_perplexity_with_context_for_secret_debug(secretKey, secrets, checkpoint_dir, dp=False, context_size=500, context_length_bin=0,
                                               prob_sampling = 1, amount_context=100):
    """Get perplexity for given examples, using random contexts.
    There are 5 bins depending on the length of the context (in tokens), 
    context_length_bin:  0 = all contexts are used
                         1 =  1-5 tokens
                         2 = 6-10 tokens
                         3 = 11-20 tokens
                         4 = 21-40 tokens
                         5 = 40+ tokens

    Args:
        secrets: a list, where each element is a secret dictionary
        checkpoint_dir: the directory from which to load checkpoints
        vocab_size: vocabulary size
        allowedVocab: Array of possible word that the model can output (masks the predicted distribution)
        dp: If set to True, load_gpt2_model loads DP-SGD optimizer instead of SGD.
        context_size: Size of context pool from where we sample one for each secret.
        context_length_bin: {0,1,2,3,4,5}, explained on previous section.
        prob_sampling: [0,1], probability of using context.
        context_amount: Amount of different contexts used for each secret.
    Returns:
        Perplexity for the input examples.
    """
    contexts = read_phrasePool("phrase_context_pool.csv", context_size, context_length_bin)
    prediction_model = load_gpt2_model("gpt2", checkpoint_dir, dp)

    sum_prob_allsecrets = 0
    secret_amount = 0
    secrets_sum_prob = 0
    for i, secret in enumerate(secrets):
        print("secret[datasets]", secret["datasets"])
        context = "MISMO CONTEXTO "
        tokenized_context = tokenizer.encode(context, add_special_tokens=False)
        context_len = len(tokenized_context)
        for r in secret["datasets"]:
            d = secret["datasets"][r]
            if (d['type'] != secretKey):
                print("Secret type", secret["datasets"][r]["type"], "found, skipping")
                break
            indexed_seqs = d['data'] 
            perplexity_for_secret = []
            for seq_batch in indexed_seqs:
                for seq in seq_batch:
                    secret_amount += 1
                    maxLength = 1024-context_len
                    full_input = tokenized_context + (seq.numpy().tolist()[:maxLength])
                    tokens = tf.convert_to_tensor([full_input]) 
                    logits = prediction_model.predict(tokens).logits
                    predictions_prob = softmax(logits, axis=-1)
                    for k in range(logits.shape[1]):
                        assert np.allclose(tf.reduce_sum(predictions_prob[0, k]).numpy(), 1.0, atol=1e-4) #Validating sum of probabilities = 1
                    
                    token_probs = 0
                    realLength = 0 #Seq has padding, repeats 2056 after the phrase ends, with RealLength we count tokens to get real phrase length
                    for k in range(context_len+1, len(full_input)):  #Skips context, hint and <eos>
                        token = full_input[k]  # Current token
                        prob = predictions_prob[0, k-1, token].numpy()  # Probability for token i given previous tokens
                        if (token == 50256):
                            break
                        token_probs -= np.log2(prob)
                        realLength += 1
 
                        

                    secret_prob = np.sqrt(2**(-token_probs)) #2 is the base of SparseCategoricalCrossentropy

                    secrets_sum_prob += secret_prob
                    print("Sumando a secrets_sum_prob",secrets_sum_prob)

    print("secret_amount", secret_amount)

    divergence = np.log2(secret_amount) - 2*np.log2(secrets_sum_prob)
    return divergence





In [ ]:
def compute_renyi_divergence_alpha_half(secretKey, secrets, checkpoint_dir, dp=False, context="MISMO CONTEXTO "):
    """Get Rényi divergence of order α=0.5 between model's distribution of secrets and uniform distribution.
    
    Args:
        secretKey: Type of secret to consider (e.g., "password", "firstNameData", etc.)
        secrets: A list of secrets, where each element is a secret dictionary
        checkpoint_dir: The directory from which to load checkpoints
        dp: If set to True, load_gpt2_model loads DP-SGD optimizer instead of SGD
        context: Context string to use when querying the model
        
    Returns:
        Rényi divergence of order α=0.5
    """
    prediction_model = load_gpt2_model("gpt2", checkpoint_dir, dp)
    tokenized_context = tokenizer.encode(context, add_special_tokens=False)
    context_len = len(tokenized_context)
    

    all_sequence_probs = []
    secret_count = 0
    for i, secret in enumerate(secrets):

        for r in secret["datasets"]:
            d = secret["datasets"][r]
            if (d['type'] != secretKey):
                print(f"Secret type {d['type']} found, skipping")
                continue
                
            indexed_seqs = d['data']
            for seq_batch in indexed_seqs:
                for seq in seq_batch:
                    secret_count += 1
                    maxLength = 1024-context_len
                    full_input = tokenized_context + (seq.numpy().tolist()[:maxLength])
                    tokens = tf.convert_to_tensor([full_input])
                    logits = prediction_model.predict(tokens).logits
                    predictions_prob = softmax(logits, axis=-1)
                    
                    log_prob = 0
                    for k in range(context_len, len(full_input)-1):
                        next_token = full_input[k+1]
                        if next_token == 50256:  # <eos> token
                            break
                        prob = predictions_prob[0, k, next_token].numpy()
                        log_prob += np.log2(prob)
                    
                    sequence_prob = 2**log_prob
                    all_sequence_probs.append(sequence_prob)
    
    print(f"Total secret count: {secret_count}")
    
    if secret_count == 0:
        print("No secrets found for the specified key.")
        return None
    
    total_prob = sum(all_sequence_probs)
    normalized_probs = [p/total_prob for p in all_sequence_probs]
    
    uniform_prob = 1.0 / secret_count
    sum_sqrt_product = sum(np.sqrt(p * uniform_prob) for p in normalized_probs)
    divergence = -2 * np.log2(sum_sqrt_product)
    
    return divergence

# Run finetuning on GPT2 with MIMIC-III dataset.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token


train_dataset, validation_dataset = get_dataset(
  filename= "NOTEEVENTS_deanonymized.csv",
   batch_size=BATCH_SIZE)


checkpoint_dir = 'DPSGD-checkpoints'
print("dataset", "loaded")
history, callbacks = train_model(train_dataset, validation_dataset, batch_size=BATCH_SIZE, vocab_size=tokenizer.vocab_size, train_epochs=TRAIN_EPOCHS, checkpoint_dir=checkpoint_dir, dp=True)

## Experiment 1:
Evaluating exposure for GPT2 finedtuned and GP2 base

In [ ]:
#1-5 Repetitions

secretsAmount = 5
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
secrets = generate_text_secrets(input_csv="secrets.txt", amount_secrets=secretsAmount, maxRepetitions = 5, minRepetitions = 1)

checkpoint_dir = 'SGD-checkpoints'
print("Calculating exposure of Finetuned model with 1 to 5 repetitions")
with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(["1 to 5 repetitions:"])


print("Calculating exposure of", "yearData")
exposure = compute_perplexity_with_context_for_secret_debug("firstNameData",
secrets["firstNameData"], checkpoint_dir, dp=False)
print("Exposure", exposure)
with open("exposureResults", mode='a', encoding='utf-8') as outfile:
        writer = csv.writer(outfile)
        writer.writerow(['Using ', checkpoint_dir,' secretKey:',"firstNameData", exposure])
   

In [ ]:
#1-5 Repetitions

contexts = read_phrasePool("phrase_context_pool.csv", 500, 0)
secretsAmount = 250
context_amount=10
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
secrets = generate_text_secrets(input_csv="secrets.txt", amount_secrets=secretsAmount, maxRepetitions = 5, minRepetitions = 1)

checkpoint_dir = 'DPSGD-checkpoints'
print("Calculating exposure of Finetuned model with 1 to 5 repetitions")
with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(["1 to 5 repetitions:"])

divergences = []

for secretKey in secrets.keys():
    if (len(secrets[secretKey]) < secretsAmount):
        print("Skipping", secretKey, "not enough repetitions:",len(secrets[secretKey]))
        continue
    for i in range(context_amount):
        ctxt =  sample_context(contexts, secretKey)["phrase"]
        print("Calculating exposure of", secretKey, "with contexts", ctxt)
        divergence = compute_renyi_divergence_alpha_half(
            secretKey=secretKey,
            secrets=secrets[secretKey],
            checkpoint_dir=checkpoint_dir,
            dp=True,
            context=ctxt
        
        )
        divergences.append(divergence)
    print("Exposure", np.mean(divergence))
    with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow([np.mean(divergence), " # Using ", checkpoint_dir,' secretKey:', secretKey])
   

In [ ]:
#1-5 Repetitions

contexts = read_phrasePool("phrase_context_pool.csv", 500, 0)
secretsAmount = 250
context_amount=10
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
secrets = generate_text_secrets(input_csv="secrets.txt", amount_secrets=secretsAmount, maxRepetitions = 5, minRepetitions = 1)

checkpoint_dir = 'SGD-checkpoints'
print("Calculating exposure of Finetuned model with 1 to 5 repetitions")
with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(["1 to 5 repetitions:"])

divergences = []

for secretKey in secrets.keys():
    if (len(secrets[secretKey]) < secretsAmount):
        print("Skipping", secretKey, "not enough repetitions:",len(secrets[secretKey]))
        continue
    if (secretKey == "doctorData") or (secretKey == "hospitalData"):
        continue

    for i in range(context_amount):
        ctxt =  sample_context(contexts, secretKey)["phrase"]
        print("Calculating exposure of", secretKey, "with contexts", ctxt)
        divergence = compute_renyi_divergence_alpha_half(
            secretKey=secretKey,
            secrets=secrets[secretKey],
            checkpoint_dir=checkpoint_dir,
            dp=False,
            context=ctxt
        
        )
        divergences.append(divergence)
    print("Exposure", np.mean(divergence))
    with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow([np.mean(divergence), " # Using ", checkpoint_dir,' secretKey:', secretKey])
   

## Evaluating variability of results with large contexts

In [ ]:
#1-5 Repetitions

contexts = read_phrasePool("phrase_context_pool.csv", 500, 0)
secretsAmount = 25
context_amount= 50
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
secrets = generate_text_secrets(input_csv="secrets.txt", amount_secrets=secretsAmount, maxRepetitions = 5, minRepetitions = 1)

checkpoint_dir = 'SGD-checkpoints'
print("Calculating exposure of Finetuned model with 1 to 5 repetitions")
with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(["1 to 5 repetitions with amount of contexts:", context_amount])

divergences = []

for secretKey in secrets.keys():
    if (len(secrets[secretKey]) < secretsAmount):
        print("Skipping", secretKey, "not enough repetitions:",len(secrets[secretKey]))
        continue
    if (secretKey not in ["doctorData", "hospitalData", "locationData", "numberData", "firstNameData" ]):
        continue

    for i in range(context_amount):
        ctxt =  sample_context(contexts, secretKey)["phrase"]
        print("Calculating exposure of", secretKey, "with contexts", ctxt)
        divergence = compute_renyi_divergence_alpha_half(
            secretKey=secretKey,
            secrets=secrets[secretKey],
            checkpoint_dir=checkpoint_dir,
            dp=False,
            context=ctxt
        
        )
        divergences.append(divergence)
    print("Exposure", np.mean(divergence))
    with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow([np.mean(divergence), " # Using ", checkpoint_dir,' secretKey:', secretKey])
   

In [ ]:
#1-5 Repetitions

contexts = read_phrasePool("phrase_context_pool.csv", 500, 0)
secretsAmount = 250
context_amount= 10
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
secrets = generate_text_secrets(input_csv="secrets.txt", amount_secrets=secretsAmount, maxRepetitions = 5, minRepetitions = 1)

checkpoint_dir = 'SGD-checkpoints'
print("Calculating exposure of Finetuned model with 1 to 5 repetitions")
with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(["1 to 5 repetitions with amount of contexts:", context_amount])



for secretKey in secrets.keys():
    divergences = []
    if (len(secrets[secretKey]) < secretsAmount):
        print("Skipping", secretKey, "not enough repetitions:",len(secrets[secretKey]))
        continue
    if (secretKey not in ["hospitalData", "locationData", "numberData", "firstNameData" ]):
        continue

    for i in range(context_amount):
        ctxt =  sample_context(contexts, secretKey)["phrase"]
        print("Calculating exposure of", secretKey, "with contexts", ctxt)
        divergence = compute_renyi_divergence_alpha_half(
            secretKey=secretKey,
            secrets=secrets[secretKey],
            checkpoint_dir=checkpoint_dir,
            dp=False,
            context=ctxt
        
        )
        divergences.append(divergence)
    print("Exposure", divergences)
    with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow([np.mean(divergences), " # Using ", checkpoint_dir,' secretKey:', secretKey])




#1-5 Repetitions

contexts = read_phrasePool("phrase_context_pool.csv", 500, 0)
secretsAmount = 250
context_amount= 10
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
secrets = generate_text_secrets(input_csv="secrets.txt", amount_secrets=secretsAmount, maxRepetitions = 5, minRepetitions = 1)

checkpoint_dir = 'DPSGD-checkpoints'
print("Calculating exposure of Finetuned model with 1 to 5 repetitions")
with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(["1 to 5 repetitions with amount of contexts:", context_amount])


for secretKey in secrets.keys():
    divergences = []
    if (len(secrets[secretKey]) < secretsAmount):
        print("Skipping", secretKey, "not enough repetitions:",len(secrets[secretKey]))
        continue
    if (secretKey not in ["doctorData", "hospitalData", "locationData", "numberData", "firstNameData" ]):
        continue

    for i in range(context_amount):
        ctxt =  sample_context(contexts, secretKey)["phrase"]
        print("Calculating exposure of", secretKey, "with contexts", ctxt)
        divergence = compute_renyi_divergence_alpha_half(
            secretKey=secretKey,
            secrets=secrets[secretKey],
            checkpoint_dir=checkpoint_dir,
            dp=True,
            context=ctxt
        
        )
        divergences.append(divergence)
    print("Exposure", np.mean(divergences))
    with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow([np.mean(divergences), " # Using ", checkpoint_dir,' secretKey:', secretKey])
   
   

1/1 [==============================] - 1s 615ms/step
Total secret count: 250
Calculating exposure of locationData with contexts FINDINGS: The major tributaries of the circle of 


All PyTorch model weights were used when initializing TFGPT2LMHeadModel.

All the weights of TFGPT2LMHeadModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


Usando DP-SGD con noise_multiplier = 5
Pesos cargados desde DPSGD-checkpoints/ckpt_4
1/1 [==============================] - 1s 632ms/step
Total secret count: 250
Calculating exposure of locationData with contexts The patient had
recently flown to 


All PyTorch model weights were used when initializing TFGPT2LMHeadModel.

All the weights of TFGPT2LMHeadModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


Usando DP-SGD con noise_multiplier = 5
Pesos cargados desde DPSGD-checkpoints/ckpt_4
1/1 [==============================] - 1s 658ms/step
Total secret count: 250
Calculating exposure of locationData with contexts Clinical Ctr 


...

Usando DP-SGD con noise_multiplier = 5
Pesos cargados desde DPSGD-checkpoints/ckpt_4
1/1 [==============================] - 1s 620ms/step
Total secret count: 250
Calculating exposure of numberData with contexts 61 yo M with Crohn's disease on prednisone s/p total colectomy
in 


All PyTorch model weights were used when initializing TFGPT2LMHeadModel.

All the weights of TFGPT2LMHeadModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


Usando DP-SGD con noise_multiplier = 5
Pesos cargados desde DPSGD-checkpoints/ckpt_4
1/1 [==============================] - 1s 636ms/step
Total secret count: 250
Calculating exposure of numberData with contexts - Hypertension
- Hyperlipidemia
- pericarditis
- Carotid artery disease (80%-[**Country **]) s/p CEA 


All PyTorch model weights were used when initializing TFGPT2LMHeadModel.

All the weights of TFGPT2LMHeadModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


Usando DP-SGD con noise_multiplier = 5
Pesos cargados desde DPSGD-checkpoints/ckpt_4
1/1 [==============================] - ETA: 0s